# Validation Test Simulation Setup - Rising Bubble Benchmark 

This notebook and the corresponding evaluation notebook (RisingBubble2D_EvaluateSingleTest.ipynb) are part of published results (cf. 7.3) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853). We compare our results against the rising bubble benchmark test case established by Hysing et al ( https://doi.org/10.1002/fld.1934)

In [1]:
//#r "BoSSSpad.dll"
#r "..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [3]:
BoSSSshell.WorkflowMgm.Init("RisingBubble2D", myBatch);

Project name is set to 'RisingBubble2D'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\RisingBubble2D'.


In [4]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [5]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [6]:
bool restartRun = false;

In [7]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_MismatchFix_restart3*	9/4/2024 2:34:52 PM	a7ac51e0...
#1: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_restart3*	9/4/2024 12:53:16 PM	d304f983...
#2: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart3*	9/2/2024 11:49:28 AM	60901f69...
#3: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_ReInit100_testcase1	8/30/2024 9:22:51 AM	44e79500...
#4: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1	8/30/2024 9:35:54 AM	46f02668...
#5: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	8/29/2024 10:31:41 AM	d3fbafe6...
#6: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withPreEnforcerActive_restart1*	8/28/2024 3:08:37 PM	d55fdec7...
#7: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	8/27/2024 2:57:24 PM	d9c4a861...


In [ ]:
//sessions.Pick(0).Delete(true)

In [22]:
//sessions.Pick(3).GetSessionDirectory()

\\dc3\userspace\smuda\hpccluster\RisingBubble2D\sessions\44e79500-d024-492e-b9ab-a46e3553b343

In [ ]:
// var restartSession = sessions.Pick(4);
// restartSession

RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...

In [ ]:
//restartSession.Timesteps.Skip(2).Export().WithSupersampling(2).Do()

In [ ]:
// string restartName = "RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_MismatchFix_restart3";

## Grid Creation

The initial setup is depicted in Figure 7, where a circular bubble (radius $r = 0.25$) is positioned at $(0.5, 0.5)$ in the domain $\Omega = [0, 1] × [0, 2]$.
The lower and upper boundary are imposed by the wall boundary condition. The boundaries at the sides describe a free-slip boundary condition given by
>$$
\vec{u} \cdot \vec{n}_{\partial \Omega} = 0, \quad \vec{\tau}_{\partial \Omega} \cdot (\nabla \vec{u} + \nabla \vec{u}^T) \vec{n}_{\partial \Omega} = 0
$$<
where $\vec{\tau}_{\partial \Omega}$ denotes the tangent vector on the boundary $\partial \Omega$.

In [8]:
bool fullDomain = true;

In [9]:
//int[] Resolutions = new int[] { 10, 20, 40, 80 };
int[] Resolutions = new int[] { 20 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"RisingBubble2D_{Res}x{2*Res}";
    GridName = fullDomain ? GridName + "_fullDomain" : "_halfDomain";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = fullDomain ? GenericBlas.Linspace(0, 1.0, Res + 1) : GenericBlas.Linspace(0.5, 1.0, Res + 1);
        double[] yNodes = GenericBlas.Linspace(0, 2.0, Res*2 + 1);
        
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;

            if (fullDomain) {
                if(X.x.Abs() <= 1e-8)
                ret = IncompressibleBcType.FreeSlip.ToString();
            } else {
                if((X.x - 0.5).Abs() <= 1e-8)
                ret = IncompressibleBcType.SlipSymmetry.ToString();
            }
            if((X.x - 1.0).Abs() <= 1e-8)
                ret = IncompressibleBcType.FreeSlip.ToString();
            if((X.y).Abs() <= 1e-8)         // lower wall
                ret = IncompressibleBcType.Wall.ToString();
            if((X.y - 2.0).Abs() <= 1e-8)   // upper wall
                ret = IncompressibleBcType.Wall.ToString();

            return ret;
        });    
        
        // grd.AddPredefinedPartitioning("FourProcSplit_fullDomain", delegate (double[] X) {
        //     int rank;
        //     double x = X[0]; double y = X[1];
        //     if (x < 0.5 && y < 0.5)
        //         rank = 0;
        //     else if (x >= 0.5 && y < 0.5)
        //         rank = 1;
        //     else if (x < 0.5 && y >= 0.5)
        //         rank = 2;
        //     else 
        //         rank = 3;

        //     return rank;
        // });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Creating database 'C:\BoSSS-localJobs\RisingBubble2D'.
Grid Edge Tags changed.
An equivalent grid (8a4977bb-5eec-43b7-aa32-77ac3e53bb09) is already present in the database -- the grid will not be saved.


## Initial Values

In [10]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double dropRadius = 0.25;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] - 0.5).Pow2() + (X[1] - 0.5).Pow2()) - dropRadius; " + 
    "}");

The driving force for the rise of the bubble ($\rho_{\mathfrak{A}} < \rho_{\mathfrak{B}}$) is the gravity force $\vec{g} = −g\vec{e}_y$
oriented in negative $y$-direction.

In [11]:
var GravityValue = new Formula(
    "grav",
    false,
    "using ilPSP.Utils; " + 
    "double grav(double[] X) { return -0.98; }");

## Physical Settings

Two test cases are defined with different physical properties resulting in an ellipsoidal terminal shape for the first test case and a dimpled cap with filaments for the second test case. The corresponding physical
parameters are given in Table 4. For both test cases the simulation is performed for three time units.

In [12]:
string testcase = "testcase1";

In [ ]:
double rhoA = 1.0;      // bubble
double rhoB = 1.0;   // surrounding fluid
double muA = 1.0;
double muB = 1.0;
double sigma = 0.0;

switch(testcase) {
    case "testcase1": { // ellipsoidal terminal shape
        rhoA = 100.0;
        rhoB = 1000.0;
        muA = 1.0;
        muB = 10.0;
        sigma = 24.5;

        break;
    }
    case "testcase2": { // dimpled cap with filaments 
        rhoA = 1.0;
        rhoB = 1000.0;
        muA = 0.1;
        muB = 10.0;
        sigma = 1.96;
        
        break;
    }
}


In [14]:
int k = 3;
int AMRlevel = 0;
double hmin = (1.0 / (double)Resolutions[0]) / ((double)AMRlevel + 1);
double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(rhoA, rhoB, sigma, hmin, k+1);
dt_sigma

0.0026731502275068254

In [15]:
double dt = 0.002;

In [16]:
int numTs = (int)(3.0/dt);
numTs

1500

## Control settings

In [17]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
//restartSession.Timesteps.Last()

 { Time-step: 1102; Physical time: 2.201999999999979s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, GravityY#A, GravityY#B, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }

In [ ]:
// int restartTS = restartSession.Timesteps.Last().TimeStepNumber.MajorNumber - 1;
// restartTS

1101

In [ ]:
for (int idx = 0; idx < Grids.Length; idx++) {

    XNSE_Control C = new XNSE_Control();

    C.SetDGdegree(k);
    C.FieldOptions.Add(VariableNames.GravityY + "#A", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });
    C.FieldOptions.Add(VariableNames.GravityY + "#B", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });

    // physical parameters
    C.PhysicalParameters.rho_A = rhoA;
    C.PhysicalParameters.mu_A = muA;
    
    C.PhysicalParameters.rho_B = rhoB;
    C.PhysicalParameters.mu_B = muB;
    
    C.PhysicalParameters.Sigma = sigma;
    
    C.PhysicalParameters.IncludeConvection = true;

    
    if (restartRun) {
        //C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, restartTS);
    } else {
        // set grid
        C.SetGrid(Grids[idx]);
    
        // initial conditions   
        C.AddInitialValue("Phi", PhiFunc);   
         
        C.AddInitialValue("GravityY#A", GravityValue);
        C.AddInitialValue("GravityY#B", GravityValue);
    }
    
    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;
 
    C.ReInitPeriod = 50;

    // C.SkipSolveAndEvaluateResidual = true;
    C.CutCellQuadratureType = BoSSS.Foundation.XDG.CutCellQuadratureMethod.OneStepGaussAndStokes;

    // C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    // C.NonLinearSolver.ConvergenceCriterion = 1e-9;
    C.NonLinearSolver.MaxSolverIterations = 50;

    // C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
    C.dtFixed = dt;
    C.NoOfTimesteps = numTs;

    C.saveperiod = 1;

    if (AMRlevel > 0) {
        C.AdaptiveMeshRefinement = true;
        C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel });
        C.AMR_startUpSweeps = AMRlevel;
    }

    C.PostprocessingModules.Add(new RisingBubble2DBenchmarkQuantities());
    // C.TracingNamespaces = \"BoSSS.Solution.AdvancedSolvers\";
   
    if (restartRun) {
        //C.SessionName = restartName;
    } else {
        int res = Resolutions[idx];
        C.SessionName = $"RB2D_fullDomain_{res}x{2*res}AMR{AMRlevel}_k{k}_Newton_StokesExt_ReInit50_{testcase}";
        //C.SessionName = $"RB3D_quarterDomain_{res}x{res}x{2*res}AMR{AMRlevel}_k{k}_{testcase}_setttingsCheck";
    }
    
    // C.GridPartType = GridPartType.Predefined;
    // C.GridPartOptions = "FourProcSplit_fullDomain";
    
    Controls.Add(C);
}

In [20]:
Controls.Select(C => C.SessionName)

[ RB2D_fullDomain_20x40AMR0_k3_Newton_StokesExt_ReInit50_testcase1 ]

## Launch Jobs

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 2;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    oneJob.UseComputeNodesExclusive = true;\n",

    ((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.Activate(myBatch);
}

 ------------ MSHPC FailedOrCanceled; original Failed
Deployments so far (1): (Job token: 93335, FailedOrCanceled 'RisingBubble2D-XNSE_Solver2024Sep04_135810.306574' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FailedOrCanceled);
Success: 0
job submit count: 1
Note: Job was deployed (1) number of times, all failed; RetryCount is 3, so try once more.
Hint: want to re-activate the job.
Job is FailedOrCanceled, but retry count is set to 3 and only 1 tries yet - trying once more...
Deploying job RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_MismatchFix_restart3 ... 
Set Database: { Session Count = 7; Grid Count = 3; Path = \\dc3\userspace\smuda\hpccluster\RisingBubble2D }
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\RisingBubble2D-XNSE_Solver2024Sep04_143321.428833
copied 42 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
